In [1]:
%pip install -q -U langchain langchain-core langgraph langgraph-checkpoint-sqlite \
                  langchain-huggingface langchain-openai transformers accelerate \
                  sentence-transformers pypdf langchain-text-splitters rank-bm25

In [2]:
"""Section-aware ingestion for the Columbia policy PDF archive.

Install requirements (if needed):
    pip install pypdf langchain-core langchain-text-splitters
"""

from __future__ import annotations

import io
import re
import unicodedata
import zipfile
from collections import defaultdict
from pathlib import PurePosixPath
from typing import Iterable

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from pypdf import PdfReader

In [10]:

# Change only this value when moving the notebook/script to another machine.
ZIP_PATH ="/content/columbia_policies_pdfs.zip"


In [3]:
from pathlib import Path
import zipfile

EXPECTED_ZIP = "columbia_policies_pdfs.zip"
SEARCH_DIRS = [
    Path.cwd(),
    Path("/content")
]


def find_policy_zip():
    for directory in SEARCH_DIRS:
        if not directory.exists():
            continue
        for candidate in directory.iterdir():
            if candidate.is_file() and candidate.name.casefold() == EXPECTED_ZIP.casefold():
                return str(candidate)

    # In Google Colab, prompt for the ZIP if it was not found in the usual locations.
    try:
        from google.colab import files
        print(f"Upload {EXPECTED_ZIP}")
        uploaded = files.upload()
        match = next(
            (name for name in uploaded if Path(name).name.casefold() == EXPECTED_ZIP.casefold()),
            None,
        )
        if match:
            return str(Path(match))
        if len(uploaded) == 1 and next(iter(uploaded)).lower().endswith(".zip"):
            only_name = next(iter(uploaded))
            print(f"Using uploaded ZIP: {only_name}")
            return str(Path(only_name))
    except ImportError:
        pass

    # Non-Colab Jupyter: enter the local path when the file is outside SEARCH_DIRS.
    entered_path = input(f"Path to {EXPECTED_ZIP}: ").strip().strip('"')
    if entered_path and Path(entered_path).is_file():
        return entered_path
    raise FileNotFoundError(f"Could not find {EXPECTED_ZIP}. Place it in the notebook folder or enter its path.")


zip_path = find_policy_zip()
with zipfile.ZipFile(zip_path, "r") as archive:
    pdf_members = [name for name in archive.namelist() if name.lower().endswith(".pdf")]

print(f"Policy ZIP: {zip_path}")
print(f"PDF files in ZIP: {len(pdf_members)}")
if not pdf_members:
    raise ValueError("The ZIP does not contain any PDF files.")

Upload columbia_policies_pdfs.zip


KeyboardInterrupt: 

In [4]:
# Repeated navigation/furniture observed in the supplied Columbia archive.
_SKIP_LINE_PATTERNS = (
    re.compile(r"^University Policies$", re.IGNORECASE),
    re.compile(r"^(?:Page\s*)?\d+\s*(?:of|/)\s*\d+$", re.IGNORECASE),
    re.compile(r"^[▻►▶➤»›]+$"),
    re.compile(r"^Copyright ©?\s*Columbia University.*$", re.IGNORECASE),
)
_TOC_LINE = re.compile(r"(?:\.{3,}|…{2,})\s*\d+\s*$")

In [5]:
# These headings occur throughout the supplied Columbia policies.  A root heading
# starts a new major policy block; a detail heading becomes its child when possible.
_ROOT_HEADINGS = {
    "policy summary",
    "overview",
    "scope",
    "exclusions & special situations",
    "exclusions and special situations",
    "policy text",
    "revision history",
    "related policies",
    "related documents",
    "appendix",
    "appendices",
}
_DETAIL_HEADINGS = {
    "purpose",
    "definitions",
    "reason for this policy",
    "reason(s) for the policy",
    "reason for policy",
    "primary guidance to which this policy responds",
    "who is governed by this policy",
    "who should know this policy",
    "roles and responsibilities",
    "responsible facilities and operations office and officer",
    "effective date",
    "prohibited conduct",
    "reporting and investigation",
    "sanctions",
    "access rights of students",
    "contact",
    "contacts",
}


In [6]:
def normalize_line(value: str) -> str:
    """Normalize PDF extraction artifacts without flattening line boundaries."""
    value = unicodedata.normalize("NFKC", value or "").replace("\u00a0", " ")
    return re.sub(r"\s+", " ", value).strip()

In [7]:
def clean_policy_lines(raw_text: str) -> list[str]:
    """Remove known PDF furniture and retain one logical extracted line per item."""
    kept: list[str] = []
    previous = None

    for raw_line in (raw_text or "").splitlines():
        line = normalize_line(raw_line)
        if not line:
            continue
        if any(pattern.fullmatch(line) for pattern in _SKIP_LINE_PATTERNS):
            continue
        if _TOC_LINE.search(line):
            continue
        # The first policy in the archive has a duplicated "Policy Text" line.
        if line == previous:
            continue
        kept.append(line)
        previous = line

    return kept


def policy_title_from_member(member: str) -> str:
    """Turn 061_Anti-Hazing_Policy.pdf into a readable metadata title."""
    stem = PurePosixPath(member).stem
    stem = re.sub(r"^\d+_", "", stem)
    return stem.replace("_", " ")

def is_title_case_heading(line: str) -> bool:
    """Conservative fallback for unnumbered headings not in the fixed list."""
    if not (2 <= len(line) <= 110) or len(line.split()) > 14:
        return False
    if re.search(r"[.!?;]$", line):
        return False
    if "," in line:
        return False

    words = re.findall(r"[A-Za-z]+", line)
    if len(words) < 2:
        return False
    capitalized = sum(word[0].isupper() for word in words)
    # Examples: "Who Is Governed By This Policy" and "Access Rights of Students".
    return capitalized / len(words) >= 0.60



def detect_heading(line: str, has_root_heading: bool):
    """Return (level, label, title), or None when a line is body text.

    Important: simple numbered list items (for example, "1. Records that ...")
    are deliberately rejected.  They remain part of their parent section.
    """
    text = normalize_line(line)
    lower = text.casefold()

    # 1. POLICY / 1.1 Transcripts / 1.1.2 Further detail
    match = re.match(
        r"^(?P<label>\d+(?:\.\d+){0,6})[.)]?\s+(?P<title>.+)$", text
    )
    if match:
        label = match.group("label")
        title = match.group("title").strip()
        # Decimal section labels are reliable.  A single-number heading must be
        # short and heading-like so ordinary numbered lists are not split apart.
        heading_like = (
            "." in label
            or (
                len(title) <= 70
                and "," not in title
                and not re.search(r"[.!?;]$", title)
                and (title.isupper() or is_title_case_heading(title))
            )
        )
        if heading_like:
            return len(label.split(".")), label, title

    # I. General Rules / A. Responsibilities
    match = re.match(r"^(?P<label>[IVXLCDM]+|[A-Z])[.)]\s+(?P<title>.+)$", text)
    if match:
        label = match.group("label")
        title = match.group("title").strip()
        if len(title) <= 100 and not re.search(r"[.!?;]$", title):
            level = 1 if re.fullmatch(r"[IVXLCDM]+", label) else 2
            return level, label, title

    # a) Procedure. / (b) Confidential Letters of Recommendation.
    match = re.match(r"^\(?(?P<label>[a-z]|[ivxlcdm]+)\)[.)]?\s*(?P<title>.+)$", text)
    if match:
        label = match.group("label")
        title = match.group("title").strip()
        # This supports FERPA's a) / (b) clauses without treating prose as a title.
        if len(title) <= 90 and len(title.split()) <= 12 and "," not in title:
            return 3 if has_root_heading else 1, label, title.rstrip(".")

    if lower in _ROOT_HEADINGS:
        return 1, "", text
    if lower in _DETAIL_HEADINGS:
        return 2 if has_root_heading else 1, "", text
    if is_title_case_heading(text):
        return 2 if has_root_heading else 1, "", text

    return None


In [8]:
def load_policy_pages(zip_path: str) -> tuple[list[Document], dict]:
    """Extract every readable page from every PDF in the ZIP archive.

    Pages are intentionally never deduplicated across PDFs: identical wording in
    two different policies is still distinct, citable policy content.
    """
    pages: list[Document] = []
    errors: list[str] = []
    pdf_count = pages_scanned = empty_pages = 0

    with zipfile.ZipFile(zip_path, "r") as archive:
        members = sorted(
            member
            for member in archive.namelist()
            if member.lower().endswith(".pdf") and not member.endswith("/")
        )
        if not members:
            raise ValueError("No PDF files were found inside the ZIP archive.")

        for member in members:
            pdf_count += 1
            try:
                reader = PdfReader(io.BytesIO(archive.read(member)))
            except Exception as exc:
                errors.append(f"{member}: could not open PDF ({exc})")
                continue

            for pdf_page, page in enumerate(reader.pages, start=1):
                pages_scanned += 1
                try:
                    lines = clean_policy_lines(page.extract_text() or "")
                except Exception as exc:
                    errors.append(f"{member}, page {pdf_page}: extraction failed ({exc})")
                    continue

                text = "\n".join(lines).strip()
                if len(text) < 25:
                    empty_pages += 1
                    continue

                pages.append(
                    Document(
                        page_content=text,
                        metadata={
                            "source": PurePosixPath(member).name,
                            "policy_title": policy_title_from_member(member),
                            "split_pdf_member": member,
                            "source_page": pdf_page,
                        },
                    )
                )

    stats = {
        "pdf_files": pdf_count,
        "pages_scanned": pages_scanned,
        "readable_pages": len(pages),
        "empty_or_boilerplate_pages_skipped": empty_pages,
        "pdf_or_page_errors": len(errors),
        "errors": errors,
    }
    return pages, stats


In [11]:




def build_section_chunks(
    policy_pages: Iterable[Document],
    chunk_size: int = 1000,
    chunk_overlap: int = 150,
) -> list[Document]:
    """Split pages at Columbia policy headings, then split oversized sections."""
    by_policy: dict[str, list[Document]] = defaultdict(list)
    for page in policy_pages:
        by_policy[page.metadata["split_pdf_member"]].append(page)

    overflow_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    output: list[Document] = []

    for member in sorted(by_policy):
        pages = sorted(
            by_policy[member], key=lambda item: item.metadata["source_page"]
        )
        policy_title = pages[0].metadata["policy_title"]
        hierarchy: dict[int, dict[str, str]] = {}
        current_heading: str | None = None
        current_body: list[str] = []
        current_pages: list[int] = []
        current_heading_page: int | None = None
        section_sequence = 0

        def emit_section() -> None:
            """Emit only sections that have real body text, not parent-only headings."""
            nonlocal current_heading, current_body, current_pages
            nonlocal current_heading_page, section_sequence

            if not current_body:
                current_heading = None
                current_pages = []
                current_heading_page = None
                return

            section_sequence += 1
            path_parts = []
            for level in sorted(hierarchy):
                item = hierarchy[level]
                path_parts.append(f'{item["label"]} {item["title"]}'.strip())
            section_path = " > ".join(path_parts) or "Preamble"

            section_text = "\n".join(
                ([current_heading] if current_heading else []) + current_body
            ).strip()
            page_numbers = sorted(set(current_pages + ([current_heading_page] if current_heading_page else [])))
            text_parts = overflow_splitter.split_text(section_text)
            deepest = hierarchy[max(hierarchy)] if hierarchy else {"label": "", "title": "Preamble"}

            for section_chunk_index, text_part in enumerate(text_parts):
                # Long sections retain their heading in every overflow chunk.
                if section_chunk_index and current_heading and not text_part.startswith(current_heading):
                    text_part = f"{current_heading}\n{text_part}"
                output.append(
                    Document(
                        page_content=text_part,
                        metadata={
                            "source": PurePosixPath(member).name,
                            "policy_title": policy_title,
                            "split_pdf_member": member,
                            # Kept for compatibility with existing retrieval/citation code.
                            "source_page": min(page_numbers),
                            "source_page_start": min(page_numbers),
                            "source_page_end": max(page_numbers),
                            "source_pages": page_numbers,
                            "section_path": section_path,
                            "section_id": deepest["label"],
                            "section_title": deepest["title"],
                            "section_sequence": section_sequence,
                            "section_chunk_index": section_chunk_index,
                            "section_chunk_count": len(text_parts),
                        },
                    )
                )

            current_heading = None
            current_body = []
            current_pages = []
            current_heading_page = None

        for page in pages:
            page_number = page.metadata["source_page"]
            for line in page.page_content.splitlines():
                heading = detect_heading(line, has_root_heading=(1 in hierarchy))
                if heading is None:
                    current_body.append(line)
                    current_pages.append(page_number)
                    continue

                emit_section()
                level, label, title = heading
                hierarchy = {
                    existing_level: item
                    for existing_level, item in hierarchy.items()
                    if existing_level < level
                }
                hierarchy[level] = {"label": label, "title": title}
                current_heading = line
                current_heading_page = page_number

        emit_section()

    for chunk_id, document in enumerate(output):
        document.metadata["chunk_id"] = chunk_id
    return output


if __name__ == "__main__":
    policy_pages, ingestion_stats = load_policy_pages(ZIP_PATH)
    if not policy_pages:
        raise ValueError("No readable policy text was extracted from the ZIP archive.")

    policy_docs = build_section_chunks(
        policy_pages,
        chunk_size=1000,
        chunk_overlap=150,
    )

    print("Ingestion summary:")
    for name, value in ingestion_stats.items():
        if name != "errors":
            print(f"  {name}: {value}")
    print(f"  retrieval_chunks: {len(policy_docs)}")

    if ingestion_stats["errors"]:
        print("\nFirst extraction errors:")
        for error in ingestion_stats["errors"][:5]:
            print(f"  - {error}")

    print("\nSample chunks:")
    for document in policy_docs[:5]:
        print(
            f'\n{document.metadata["policy_title"]} | '
            f'pages {document.metadata["source_page_start"]}-'
            f'{document.metadata["source_page_end"]}'
        )
        print(f'Section: {document.metadata["section_path"]}')
        print(document.page_content[:350], "...")


Ingestion summary:
  pdf_files: 279
  pages_scanned: 1409
  readable_pages: 1385
  empty_or_boilerplate_pages_skipped: 24
  pdf_or_page_errors: 0
  retrieval_chunks: 5693

Sample chunks:

About Columbia | pages 1-1
Section: University Mission Statement
University Mission Statement
Columbia University is one of the world’s most important centers of research and at the same time a
distinctive and distinguished learning environment for undergraduates and graduate students in
many scholarly and professional fields. The University recognizes the importance of its location in
New York City and seeks to ...

About Columbia | pages 1-2
Section: University Mission Statement > About »Home
About »Home
For more than 270 years, Columbia alumni, faculty, staff, and students have made major
contributions to education, science and technology research, the arts, our city, our nation, and the
world. ...

About Columbia | pages 2-2
Section: University Mission Statement > Explore Columbia's Impact
Explore

In [16]:
for document in policy_docs[10:20]:
        print(
            f'\n{document.metadata["policy_title"]} | '
            f'pages {document.metadata["source_page_start"]}-'
            f'{document.metadata["source_page_end"]}'
        )
        print(f'Section: {document.metadata["section_path"]}')
        print(document.page_content[:350], "...")


About Columbia | pages 4-4
Section: University Mission Statement > Campus Access Advisory Committee
Campus Access Advisory Committee
We are working continuously to fulfill our
commitments and implement these reforms. ...

About Columbia | pages 4-4
Section: University Mission Statement > Combating Antisemitism
Combating Antisemitism
The University has zero tolerance for
discrimination and harassment based on
protected traits. ...

About Columbia | pages 5-5
Section: University Mission Statement > Our History
Our History
Academics ...

About Columbia | pages 5-6
Section: University Mission Statement > Campus Life
Campus Life
Research ...

About Columbia | pages 6-6
Section: University Mission Statement > Columbia News
Columbia News
Athletics ...

About Columbia | pages 6-6
Section: University Mission Statement > UPCOMING EVENTS
UPCOMING EVENTS
There are no upcoming events. ...

Retention of Student Education Records | pages 1-1
Section: Policy Summary
Policy Summary
This policy describ

In [ ]:
# saving the chunks

In [17]:
import json

output_file = "columbia_policy_retrieval_chunks.jsonl"

with open(output_file, "w", encoding="utf-8") as file:
    for doc in policy_docs:
        file.write(json.dumps({
            "page_content": doc.page_content,
            "metadata": doc.metadata,
        }, ensure_ascii=False) + "\n")

print(f"Saved {len(policy_docs)} chunks to: {output_file}")

Saved 5693 chunks to: columbia_policy_retrieval_chunks.jsonl


In [ ]:
# reading them back

In [18]:
import json
from langchain_core.documents import Document

with open("columbia_policy_retrieval_chunks.jsonl", "r", encoding="utf-8") as file:
    policy_docs = [
        Document(
            page_content=item["page_content"],
            metadata=item["metadata"],
        )
        for item in map(json.loads, file)
    ]

print(f"Loaded {len(policy_docs)} chunks")

Loaded 5693 chunks
